# Trend Tracker — Notebook 02: Semantic Enrichment

Builds two injection maps from the preprocessed vocabulary, then writes an
enriched parquet that `03_insights_generation` can load in place of `05_filtered.parquet`.

**Run order:** `01_token_preprocess` → `02_semantic_enrichment` → `03_insights_generation`

| Pass | What | Key outputs |
|------|------|-------------|
| A | Subject clustering (SentenceTransformer + agglomerative + LLM gate) | `enrichment/semantic_map.csv` |
| B | Framing taxonomy classification (LLM batch) | `enrichment/framing_map.csv` |
| C | Token injection | `prepared/05_enriched.parquet` |

**Recommended spot checks:**
- After Pass A: review `semantic_map.csv` — term → subject category mappings.
- Before Pass B: confirm `framing_taxonomy.json` is the intended taxonomy version.
- After Pass B: review `framing_map.csv` and any coverage deviation warnings.
- After Pass C: check injection stats and decide whether to use the enriched or filtered parquet downstream.

---
## Setup and Configuration

In [1]:
import os
import sys
import json
import yaml
import hashlib
import pickle
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import httpx
import urllib3

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

import importlib.util as _ilu, sys as _sys
_u = next(Path(".").glob("utils*.py"))
if _u.stem != "utils":
    _s = _ilu.spec_from_file_location("utils", _u); _m = _ilu.module_from_spec(_s); _s.loader.exec_module(_m); _sys.modules["utils"] = _m
    
from utils import (
    load_cfg,
    flat_freq,
    build_output_path,
    get_llm_client,
    ensure_warning_file,
    append_warning,
    start_stage_manifest,
    finalize_stage_manifest,
    artifact_meta,
    gate_cluster,
    classify_batch,
    inject_tokens,
    resolve_params_path,
)

CFG_PATH = resolve_params_path()
CFG = load_cfg(CFG_PATH)

try:
    MANIFEST_CFG_PATH = str(CFG_PATH.relative_to(ROOT))
except ValueError:
    MANIFEST_CFG_PATH = str(CFG_PATH)

# Convenience wrapper: NB02 outputs are not run-scoped.
def OUT(subdir, fname):
    return build_output_path(subdir, fname)

# ── Proxy environment: SSL verification bypass ───────────────────────────────
# SentenceTransformer downloads models via the HuggingFace Hub (httpx +
# requests). The DonorsChoose proxy requires SSL verification to be disabled
# for outbound model downloads. This patch is idempotent and only runs once
# per kernel session.
os.environ["CURL_CA_BUNDLE"]    = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""

if not getattr(httpx.Client, "_dc_verify_patched", False):
    _orig_init = httpx.Client.__init__
    def _patched_init(self, *args, **kwargs):
        kwargs["verify"] = False
        return _orig_init(self, *args, **kwargs)
    httpx.Client.__init__ = _patched_init
    httpx.Client._dc_verify_patched = True

urllib3.disable_warnings()

# ── Warnings, manifests, LLM client ──────────────────────────────────────────
WARNINGS_PATH = OUT("enrichment", "warnings_02.jsonl")
ensure_warning_file(WARNINGS_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="02_semantic_enrichment",
    notebook_file="02_semantic_enrichment_v1.ipynb",
    config_path=MANIFEST_CFG_PATH,
    run_id=None,
    group_by_field=None,
    filter_fields_key=None,
)

client         = get_llm_client()
MODEL_GATE     = CFG["models"]["gate"]
MODEL_CLASSIFY = CFG["models"]["classify"]

# ── Load corpus and vocabulary ────────────────────────────────────────────────
df       = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_filtered.parquet")
doc_freq = (
    pd.read_csv(ROOT / "OUTPUTS/vocab/unigram_docfreq.csv", index_col=0)
    .squeeze()
    .rename("doc_freq")
)
freq     = flat_freq(df).rename("corpus_freq")
vocab_df = pd.DataFrame({"corpus_freq": freq, "doc_freq": doc_freq}).dropna()
vocab_df["docs_pct"] = vocab_df["doc_freq"] / len(df)

print(f"Corpus     : {len(df):,} projects")
print(f"Full vocab : {len(vocab_df):,} terms")
print(f"Warnings   : {WARNINGS_PATH}")

Corpus     : 2,244,916 projects
Full vocab : 7,554 terms
Warnings   : /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/enrichment/warnings_02.jsonl


---
## Pass A — Subject Clustering

Embeds rare-vocabulary terms with SentenceTransformer, clusters them with
agglomerative cosine-distance clustering, then runs each cluster through an
LLM coherence gate.

Gate decisions:
- **inject** — terms share a coherent educational domain concept; a `__cat_*__`
  token is added to any project containing a member term.
- **split** — 2–3 coherent sub-groups exist; each sub-group is treated as a
  separate inject.
- **discard** — generic language, morphological noise, or no meaningful shared
  concept.

Embeddings are cached by input hash so re-runs skip the encode step.

In [2]:
RARE_DOC_FREQ_CEILING  = CFG["enrichment"]["rare_doc_freq_ceiling"]
AGG_DISTANCE_THRESHOLD = CFG["enrichment"]["agg_distance_threshold"]
MIN_CLUSTER_SIZE       = CFG["enrichment"]["min_cluster_size"]
EMBEDDING_MODEL        = CFG["enrichment"]["embedding_model"]
CACHE_SCHEMA_VERSION   = CFG["enrichment"]["cache_schema_version"]

# ── Select rare terms to cluster ──────────────────────────────────────────────
rare_terms = vocab_df[vocab_df["doc_freq"] < RARE_DOC_FREQ_CEILING].index.tolist()
print(f"Rare terms to cluster: {len(rare_terms):,}")

# ── Build a cache key from inputs ─────────────────────────────────────────────
# Any change to the term list, model, thresholds, or schema version produces
# a new key and forces a fresh encode + cluster run.
cache_components = "|".join([
    hashlib.md5(",".join(sorted(rare_terms)).encode()).hexdigest(),
    EMBEDDING_MODEL,
    str(AGG_DISTANCE_THRESHOLD),
    str(MIN_CLUSTER_SIZE),
    CACHE_SCHEMA_VERSION,
])
cache_key  = hashlib.md5(cache_components.encode()).hexdigest()
CACHE_DIR  = ROOT / ".cache"
CACHE_DIR.mkdir(exist_ok=True)
cache_path = CACHE_DIR / f"embeddings_clusters_{cache_key[:12]}.pkl"

# ── Guard: skip Pass A when too few rare terms exist ────────────────────────
# AgglomerativeClustering requires at least 2 samples. When rare_terms is empty
# or contains only 1 term — possible in narrow or heavily filtered runs — the
# clustering call would raise a ValueError. We short-circuit here and produce
# empty-but-valid outputs so downstream cells continue without error.
if len(rare_terms) < 2:
    append_warning(
        WARNINGS_PATH, "02_semantic_enrichment", "PASS_A_SKIPPED",
        f"Pass A skipped: only {len(rare_terms)} rare term(s) below ceiling "
        f"(minimum 2 required for clustering).",
        context={"rare_term_count": len(rare_terms), "ceiling": RARE_DOC_FREQ_CEILING},
    )
    print(
        f"Pass A skipped — only {len(rare_terms)} rare term(s) found below "
        f"doc_freq ceiling {RARE_DOC_FREQ_CEILING}. "
        "Minimum 2 required for clustering."
    )
    # Produce empty-but-valid outputs so the gate and semantic_map cells continue.
    clusters = pd.DataFrame(columns=["cluster_id", "terms"])

# ── Embed and cluster (or load from cache) ────────────────────────────────────
elif cache_path.exists():
    print(f"Cache hit ({cache_key[:8]}) — loading embeddings and cluster labels...")
    with open(cache_path, "rb") as f:
        embeddings, labels = pickle.load(f)
else:
    print(f"Cache miss — embedding {len(rare_terms):,} terms with {EMBEDDING_MODEL}...")
    embedder   = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = embedder.encode(
        rare_terms, show_progress_bar=True, normalize_embeddings=True
    )
    agg = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=AGG_DISTANCE_THRESHOLD,
        metric="cosine",
        linkage="average",
    )
    labels = agg.fit_predict(embeddings)
    with open(cache_path, "wb") as f:
        pickle.dump((embeddings, labels), f)
    print(f"Written to cache: {cache_path.name}")

# ── Filter to valid clusters and inspect ─────────────────────────────────────
sizes     = Counter(labels)
valid_ids = {cid for cid, n in sizes.items() if n >= MIN_CLUSTER_SIZE}
cluster_df = pd.DataFrame({"term": rare_terms, "cluster_id": labels})
cluster_df = cluster_df[cluster_df["cluster_id"].isin(valid_ids)].copy()
clusters   = (
    cluster_df.groupby("cluster_id")["term"].apply(list).reset_index(name="terms")
)

print(f"\nTerms in valid clusters : {len(cluster_df):,} of {len(rare_terms):,}")
print(f"Valid clusters ({MIN_CLUSTER_SIZE}+ members): {len(clusters)}")
print("\nSample clusters:")
for _, row in clusters.sample(min(8, len(clusters)), random_state=42).iterrows():
    print(f"  C{row.cluster_id:03d}: {row.terms[:12]}")

Rare terms to cluster: 2,432
Cache hit (aa5a76c9) — loading embeddings and cluster labels...

Terms in valid clusters : 94 of 2,432
Valid clusters (3+ members): 30

Sample clusters:
  C315: ['advertise', 'advertisement', 'advertising']
  C096: ['candidate', 'nominate', 'nominee']
  C201: ['gumball', 'gummies', 'gummy']
  C114: ['el', 'elapse', 'elar', 'elate']
  C049: ['footstep', 'footwear', 'footwork']
  C055: ['aunt', 'cousin', 'uncle']
  C442: ['battleship', 'naval', 'navy']
  C202: ['ios', 'iphone', 'iphones']


In [3]:
# ── LLM coherence gate ────────────────────────────────────────────────────────
# One API call per cluster. gate_cluster() is defined in utils.py and handles
# retry / error logic. Prompts are defined here to keep domain language visible.

GATE_SYSTEM = (
    "You are evaluating vocabulary clusters extracted from DonorsChoose K-12 teacher "
    "project essays for a semantic enrichment pipeline.\n\n"
    "Your decisions control which clusters become injected enrichment tokens "
    "(for example: __cat_marine_biology__). These injected tokens are appended to "
    "project token lists and then used downstream as TF-IDF and NMF signals in topic "
    "modeling.\n\n"
    "Precision matters more than recall. A false inject approves a noisy, generic, or "
    "mixed cluster and pollutes downstream topics with meaningless signal. A false "
    "discard is acceptable. Under-injection is cheaper than over-injection.\n\n"
    "Use short snake_case labels when labeling coherent concepts.\n"
    "Return ONLY valid JSON. No markdown, no code fences, no extra keys."
)

GATE_PROMPT = (
    "Cluster {cid} terms: {terms}\n\n"
    "Decide whether these terms form a coherent, specific educational domain concept "
    "worth surfacing as an injected topic-modeling signal.\n\n"
    "Actions:\n"
    "inject:\n"
    "- Use only when a clear majority of terms cohere around one specific concept.\n"
    "- Any outlier terms should be plausibly related near-neighbors, not unrelated noise.\n"
    "- If you would need to stretch to explain how more than 2-3 terms fit the label, "
    "do not inject.\n"
    "- Assign a short snake_case primary_category.\n"
    "- Good granularity is roughly: marine_biology, sign_language, woodworking, "
    "circuit_building.\n"
    "- Too broad: biology, art, reading.\n"
    "- Too narrow: opisthokont_phylogeny.\n"
    "- subcategory: optional refinement of primary_category when a narrower, still-useful "
    "concept is clearly supported by the terms "
    "(e.g. primary_category=marine_biology, subcategory=coral_reef_ecology). "
    "Otherwise use null.\n\n"
    "split:\n"
    "- Use only when the cluster clearly contains 2-3 distinct coherent subgroups.\n"
    "- Each subgroup should have at least 3 terms.\n"
    "- You must assign the original terms into subgroup term lists.\n"
    "- If subgroup boundaries are weak or debatable, discard instead.\n\n"
    "discard:\n"
    "- Use for generic instructional language, weak thematic overlap, mixed concepts, "
    "morphological variants, noisy tokens, or overlapping senses.\n"
    "- When in doubt, discard.\n\n"
    "Examples:\n"
    "[inject] [terrarium, vivarium, reptile_habitat, snake_enclosure] "
    '-> {{"action":"inject","primary_category":"terrarium_keeping","subcategory":null,'
    '"split_into":[],"reasoning":"The terms consistently point to one specific habitat-keeping concept."}}\n'
    "[inject] [coral, reef_tank, anemone, clownfish, salinity] "
    '-> {{"action":"inject","primary_category":"marine_biology","subcategory":"coral_reef_ecology",'
    '"split_into":[],"reasoning":"The terms point to a specific marine-biology subdomain centered on reef ecosystems."}}\n'
    "[split] [watercolor, oil_paint, acrylic, welding, metalwork, soldering] "
    '-> {{"action":"split","primary_category":null,"subcategory":null,"split_into":['
    '{{"label":"painting_media","terms":["watercolor","oil_paint","acrylic"]}},'
    '{{"label":"metal_fabrication","terms":["welding","metalwork","soldering"]}}'
    '],"reasoning":"The cluster contains two clear craft/material subdomains."}}\n'
    "[discard] [learning, skill, activity, practice, work] "
    '-> {{"action":"discard","primary_category":null,"subcategory":null,"split_into":[],'
    '"reasoning":"The terms are generic instructional language rather than one specific concept."}}\n'
    "[discard] [un_funded, re_funded, pre_funded] "
    '-> {{"action":"discard","primary_category":null,"subcategory":null,"split_into":[],'
    '"reasoning":"The cluster is morphological noise rather than a coherent domain concept."}}\n\n'
    "Return a JSON object with exactly these keys:\n"
    '  "action": "inject" | "split" | "discard"\n'
    '  "primary_category": snake_case string or null\n'
    '  "subcategory": snake_case string or null\n'
    '  "split_into": [{{"label": "snake_case_label", "terms": ["term1", "term2", ...]}}, ...] '
    'if action="split", else []\n'
    '  "reasoning": one short sentence grounded in the terms\n\n'
    "Output constraints:\n"
    "- If action = inject, provide primary_category; use subcategory only if it adds useful specificity.\n"
    "- If action = split, primary_category=null and subcategory=null.\n"
    "- If action = discard, set primary_category=null, subcategory=null, split_into=[].\n"
    "- Return JSON only."
)

gate_results = []
for i, row in clusters.iterrows():
    result = gate_cluster(
        row.cluster_id, row.terms,
        client=client,
        model=MODEL_GATE,
        system_prompt=GATE_SYSTEM,
        prompt_template=GATE_PROMPT,
        retries=CFG["llm"]["max_retries"],
    )
    result["cluster_id"] = row.cluster_id
    result["terms"]      = row.terms
    gate_results.append(result)
    action = result.get("action", "?")
    label  = result.get("primary_category", "")
    if (i % 10 == 0) or action != "discard":
        print(f"  C{row.cluster_id:03d} [{action:7s}] {label or result.get('reasoning','')[:60]}")

# Schema-stable even when no clusters were processed.
GATE_SCHEMA = ["cluster_id", "terms", "action", "primary_category", "subcategory", "split_into", "reasoning"]
gate_df = (
    pd.DataFrame(gate_results)
    if gate_results
    else pd.DataFrame(columns=GATE_SCHEMA)
)
print("\nGate summary:")
if not gate_df.empty:
    print(gate_df["action"].value_counts().to_string())
else:
    print("  No clusters were gated (pass produced no results).")


  C002 [inject ] latin_american_countries
  C009 [inject ] family_relatives
  C016 [inject ] renovation
  C062 [inject ] european_countries
  C079 [inject ] dodgeball
  C150 [inject ] audio_story_player
  C196 [inject ] textile_apparel
  C202 [inject ] apple_mobile_devices
  C315 [inject ] advertising
  C442 [inject ] naval_warfare

Gate summary:
action
discard    20
inject     10


In [4]:
# ── Expand gate results into a flat term → category mapping ───────────────────
sem_rows = []

for _, row in gate_df.iterrows():
    if row.action == "inject" and row.primary_category:
        for term in row.terms:
            sem_rows.append({
                "term": term,
                "primary_category": row.primary_category,
                "subcategory": row.get("subcategory"),
                "source_cluster": row.cluster_id,
            })
    elif row.action == "split" and row.get("split_into"):
        for sub in row["split_into"]:
            for term in sub.get("terms", []):
                sem_rows.append({
                    "term": term,
                    "primary_category": sub["label"],
                    "subcategory": None,
                    "source_cluster": row.cluster_id,
                })

# Schema-stable even when no terms were accepted.
SEMANTIC_MAP_SCHEMA = ["term", "primary_category", "subcategory", "source_cluster"]
semantic_map = (
    pd.DataFrame(sem_rows)
    if sem_rows
    else pd.DataFrame(columns=SEMANTIC_MAP_SCHEMA)
)
semantic_map.to_csv(OUT("enrichment", "semantic_map.csv"), index=False)

print(f"semantic_map.csv: {len(semantic_map):,} terms mapped")
if not semantic_map.empty:
    print(f"Unique primary categories: {semantic_map['primary_category'].nunique()}")
    print("\nTop categories by term count:")
    print(semantic_map["primary_category"].value_counts().head(20).to_string())
else:
    print("No terms mapped — semantic_map.csv written with headers only.")

semantic_map.csv: 33 terms mapped
Unique primary categories: 10

Top categories by term count:
primary_category
european_countries          6
latin_american_countries    3
family_relatives            3
renovation                  3
dodgeball                   3
audio_story_player          3
textile_apparel             3
apple_mobile_devices        3
advertising                 3
naval_warfare               3


---
## Pass B — Framing Taxonomy Classification

Classifies the full vocabulary against a curated framing taxonomy to identify
terms that carry rhetorical tone or contextual framing signals (urgency,
gratitude, scarcity, etc.).

**Taxonomy source:** `OUTPUTS/enrichment/framing_taxonomy.json` (curated externally).
To update the category set, edit that file directly and set
`enrichment.force_reclassify: true` in `params.yaml` before re-running.

Classification results are cached by taxonomy hash, vocabulary hash, model, and
batch size. The cache is invalidated automatically when any of these change;
`force_reclassify: true` overrides the cache regardless.

In [5]:
# ── Load curated framing taxonomy ────────────────────────────────────────────
# The taxonomy is developed and maintained externally. Copy the current version
# to OUTPUTS/enrichment/framing_taxonomy.json before running this cell.
# To update categories, edit the file and set force_reclassify: true.

taxonomy_path = OUT("enrichment", "framing_taxonomy.json")
if not taxonomy_path.exists():
    raise FileNotFoundError(
        f"framing_taxonomy.json not found at {taxonomy_path}\n"
        "Copy your curated taxonomy file to OUTPUTS/enrichment/ before running."
    )

with open(taxonomy_path) as f:
    taxonomy = json.load(f)

nmf_cats      = [c for c in taxonomy if c.get("tier") == "nmf"]
analysis_cats = [c for c in taxonomy if c.get("tier") == "analysis"]

print(f"Loaded {len(taxonomy)} categories from {taxonomy_path.name}")
print(f"  NMF-tier:      {len(nmf_cats)}")
print(f"  Analysis-tier: {len(analysis_cats)}")
print()
for cat in taxonomy:
    tier = cat.get("tier", "?")
    cov  = cat.get("target_coverage_pct", "?")
    print(f"  [{tier:8s}] {cat['category']:50s} {cov}")

Loaded 81 categories from framing_taxonomy.json
  NMF-tier:      15
  Analysis-tier: 66

  [nmf     ] episodic_disruption_recovery                       20-35%
  [nmf     ] chronic_scarcity                                   18-32%
  [nmf     ] barrier_removal                                    25-40%
  [nmf     ] catch_up_remediation                               18-30%
  [nmf     ] opportunity_expansion                              20-35%
  [nmf     ] capacity_first_dignity                             18-30%
  [nmf     ] regulation_safety_need                             18-32%
  [nmf     ] readiness_to_learn                                 20-35%
  [nmf     ] loss_avoidance                                     18-32%
  [nmf     ] urgent_now_anchoring                               15-25%
  [nmf     ] future_preparation                                 18-30%
  [nmf     ] household_basic_needs_spillover                    15-25%
  [nmf     ] inclusion_accommodation_framing               

In [7]:
BATCH_SIZE       = CFG["enrichment"]["classify_batch_size"]
FORCE_RECLASSIFY = CFG["enrichment"]["force_reclassify"]
framing_map_path  = OUT("enrichment", "framing_map.csv")
framing_meta_path = OUT("enrichment", "framing_map_meta.json")

# ── Cache keys for this run ───────────────────────────────────────────────────
with open(OUT("enrichment", "framing_taxonomy.json")) as _f:
    current_taxonomy_hash = hashlib.md5(_f.read().encode()).hexdigest()

# vocab_md5 captures the current vocabulary so a corpus change (e.g. re-running
# NB01 with different thresholds) invalidates the cache even when the taxonomy
# file is unchanged.
vocab_md5 = hashlib.md5(",".join(sorted(vocab_df.index.tolist())).encode()).hexdigest()

def _meta_hash_matches():
    """Return True if the cached framing_map matches all current inputs.

    Validates taxonomy version, vocabulary, model, and batch size so that
    any change to the corpus or classification config invalidates the cache.
    """
    if not framing_meta_path.exists():
        return False
    with open(framing_meta_path) as _mf:
        meta = json.load(_mf)
    return (
        meta.get("taxonomy_md5")           == current_taxonomy_hash
        and meta.get("vocab_md5")          == vocab_md5
        and meta.get("model")              == MODEL_CLASSIFY
        and meta.get("classify_batch_size") == BATCH_SIZE
    )

# ── Build taxonomy reference string and valid category set for prompts ────────
def _fmt_cat(cat):
    return "\n".join([
        f"  [{cat.get('tier','?')}] {cat['category']} (target: {cat.get('target_coverage_pct','?')})",
        f"    Description: {cat['description']}",
        f"    Examples: {', '.join(cat.get('examples', [])[:8])}",
        f"    Counter-examples (must return null): {', '.join(cat.get('counter_examples', [])[:6])}",
    ])

taxonomy_ref     = "\n\n".join(_fmt_cat(cat) for cat in taxonomy)
valid_categories = {cat["category"] for cat in taxonomy}

CLASSIFY_SYSTEM = (
    "You classify vocabulary terms from DonorsChoose K-12 teacher project essays into a "
    "closed set of framing and tone categories.\n\n"
    "Your output drives injection of __framing_*__ tokens into project token lists. "
    "Those injected tokens become downstream topic-modeling signals that capture "
    "rhetorical register and emotional framing, separate from subject matter.\n\n"
    "Core distinction:\n"
    "- Classify terms that signal HOW a teacher is framing a request "
    "(for example: urgency, gratitude, scarcity, hope, burden, aspiration).\n"
    "- Do not classify terms that primarily signal WHAT is being requested "
    "(for example: microscope, pencils, chromebook, lab table).\n\n"
    "Most terms should be null.\n"
    "Classify a term only when its framing meaning is evident without needing sentence "
    "context — you could label it correctly knowing only the word.\n\n"
    "Use only the supplied taxonomy categories. Never invent a category. Prefer null "
    "over a weak guess.\n\n"
    "Return ONLY a valid JSON object mapping every input term to either one exact "
    "taxonomy category name or null. No markdown, no code fences, no commentary."
)

CLASSIFY_PROMPT = (
    "Classify each term into exactly one taxonomy category or null.\n\n"
    "Expected sparsity:\n"
    "- Only a small minority of terms should be classified.\n"
    "- A typical batch of this size may yield roughly 5-20 classified terms; the rest "
    "should be null.\n"
    "- Err heavily toward null.\n\n"
    "Classify a term only when ALL of the following are true:\n"
    "1. The term signals tone, stance, or rhetorical framing rather than subject matter.\n"
    "2. The framing meaning is evident without needing sentence context.\n"
    "3. The term clearly fits one taxonomy category.\n\n"
    "Use null when the term is:\n"
    "- subject-matter vocabulary\n"
    "- generic school or classroom language\n"
    "- merely descriptive without rhetorical force\n"
    "- ambiguous across multiple categories\n"
    "- dependent on sentence context to carry framing meaning\n\n"
    "Framing categories:\n"
    "{taxonomy}\n\n"
    "Reminder: classify only if the term signals HOW the teacher frames the request, "
    "not WHAT is being requested.\n\n"
    "Terms: {terms}\n\n"
    "Return exactly one JSON object of this form:\n"
    '{{"term1": "exact_taxonomy_category_name", "term2": null}}\n'
    "Include every input term as a key. The example above shows only two keys for brevity.\n\n"
    "Requirements:\n"
    "- Every input term must appear exactly once as a key.\n"
    "- Use only exact category names from the taxonomy.\n"
    "- Do not explain your answers.\n"
    "- Return JSON only."
)

# ── Classify or load from cache ───────────────────────────────────────────────
if not FORCE_RECLASSIFY and framing_map_path.exists() and _meta_hash_matches():
    framing_map = pd.read_csv(framing_map_path)
    print(f"Cache hit — reusing framing_map.csv ({len(framing_map):,} terms classified)")
else:
    if framing_map_path.exists() and not _meta_hash_matches():
        print("Cache invalid — re-classifying (taxonomy, vocab, model, or batch size changed).")

    all_terms   = vocab_df.index.tolist()
    batches     = [all_terms[i:i + BATCH_SIZE] for i in range(0, len(all_terms), BATCH_SIZE)]
    framing_raw = {}

    print(f"Classifying {len(all_terms):,} terms in {len(batches)} batches...")
    for i, batch in enumerate(batches):
        result = classify_batch(
            batch,
            client=client,
            model=MODEL_CLASSIFY,
            system_prompt=CLASSIFY_SYSTEM,
            prompt_template=CLASSIFY_PROMPT,
            taxonomy_ref=taxonomy_ref,
            valid_categories=valid_categories,
            retries=CFG["llm"]["max_retries"],
        )
        framing_raw.update(result)
        if (i + 1) % 10 == 0:
            classified = sum(1 for v in framing_raw.values() if v is not None)
            print(f"  Batch {i+1}/{len(batches)} — {classified:,} terms classified so far")

    framing_map = pd.DataFrame([
        {"term": term, "framing_category": cat}
        for term, cat in framing_raw.items()
        if cat is not None
    ])
    framing_map.to_csv(framing_map_path, index=False)

# ── Persist provenance metadata ───────────────────────────────────────────────
meta = {
    "taxonomy_md5":        current_taxonomy_hash,
    "vocab_md5":           vocab_md5,
    "model":               MODEL_CLASSIFY,
    "classify_batch_size": BATCH_SIZE,
    "classified_at":       datetime.now().isoformat(),
    "n_terms_classified":  int(len(framing_map)),
    "n_categories":        int(framing_map["framing_category"].nunique()) if len(framing_map) else 0,
    "coverage_pct":        round(len(framing_map) / len(vocab_df) * 100, 2) if len(vocab_df) else 0.0,
}
with open(framing_meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"\nframing_map.csv : {len(framing_map):,} terms classified")
print(f"Vocab coverage  : {len(framing_map)/len(vocab_df):.1%}")
if not framing_map.empty:
    print("\nCategory distribution:")
    print(framing_map["framing_category"].value_counts().to_string())
else:
    print("No terms classified — framing_map.csv is empty.")

# ── Coverage deviation warnings ───────────────────────────────────────────────
# Warn when a category's realized coverage deviates from its taxonomy target by
# more than coverage_deviation_warn_threshold percentage points.
warn_threshold    = CFG["enrichment"]["coverage_deviation_warn_threshold"]
actual_pct_by_cat = (
    framing_map["framing_category"].value_counts(normalize=True).mul(100).to_dict()
    if not framing_map.empty
    else {}
)

def _target_midpoint(v):
    """Parse a target coverage string like '2-4%' to its numeric midpoint."""
    if v is None:
        return None
    s = str(v).strip().replace("%", "")
    if "-" in s:
        lo, hi = s.split("-", 1)
        try:
            return (float(lo) + float(hi)) / 2
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

for cat in taxonomy:
    target = _target_midpoint(cat.get("target_coverage_pct"))
    if target is None:
        continue
    actual = actual_pct_by_cat.get(cat["category"], 0.0)
    if abs(actual - target) > warn_threshold:
        append_warning(
            WARNINGS_PATH, "02_semantic_enrichment", "COVERAGE_DEVIATION",
            f"Coverage deviation for {cat['category']}",
            context={"category": cat["category"], "actual_pct": actual, "target_pct": target},
        )
        print(
            f"WARNING: {cat['category']}: "
            f"actual={actual:.1f}% vs target\u2248{target:.1f}% "
            f"(threshold \u00b1{warn_threshold}pp)"
        )


Cache invalid — re-classifying (taxonomy, vocab, model, or batch size changed).
Classifying 7,554 terms in 51 batches...
  Batch 10/51 — 441 terms classified so far
  Batch 20/51 — 812 terms classified so far
  Batch 30/51 — 1,162 terms classified so far
  Batch 40/51 — 1,465 terms classified so far
  Batch 50/51 — 1,854 terms classified so far

framing_map.csv : 1,862 terms classified
Vocab coverage  : 24.6%

Category distribution:
framing_category
capacity_first_dignity                       127
joy_possibility_appeal                       123
foundational_communication_literacy           74
self_regulation_executive_function            48
causal_specificity                            48
mental_health_wellbeing_support               47
safety_protection_language                    47
inclusion_accommodation_framing               46
household_basic_needs_spillover               43
expressive_creative_production                40
sensory_detail                                39
durable

---
## Pass C — Token Injection

Builds an injection lookup from Pass A and Pass B results, then appends
enrichment tokens to each project's token list. Injection is strictly
additive — original tokens are never modified or removed.

Token naming convention:
- `__cat_marine_biology__` — subject category (Pass A inject)
- `__sub_ocean_ecosystems__` — subject subcategory (Pass A split)
- `__framing_urgency__` — framing signal (Pass B)

The double-underscore prefix ensures injected tokens are visually distinct
in all downstream outputs and can be filtered or analyzed independently.

In [8]:
# ── Load injection config ─────────────────────────────────────────────────────
# Ceiling exceptions are high-frequency terms that bypass max_doc_freq_for_injection
# because they are definitionally load-bearing for their framing category.
# Source: params.yaml → enrichment.ceiling_exceptions_path
ceiling_path = ROOT / CFG["enrichment"]["ceiling_exceptions_path"]
with open(ceiling_path) as f:
    ceiling_exceptions = yaml.safe_load(f)
print(f"Ceiling exceptions : {len(ceiling_exceptions)} terms from {ceiling_path.name}")

MIN_INJECT_PROJECTS     = CFG["enrichment"]["min_inject_projects"]
MAX_DOC_FREQ_FOR_INJECT = CFG["enrichment"]["max_doc_freq_for_injection"]

# ── Reload maps from disk ─────────────────────────────────────────────────────
semantic_map = pd.read_csv(OUT("enrichment", "semantic_map.csv"))
framing_map  = pd.read_csv(OUT("enrichment", "framing_map.csv"))

# Reload valid_categories from the on-disk taxonomy so Pass C is self-contained
# and does not depend on the Pass B in-memory variable. This makes it safe to
# re-run Pass C independently without re-running Pass B first.
with open(OUT("enrichment", "framing_taxonomy.json")) as _f:
    _tax = json.load(_f)
valid_categories = {cat["category"] for cat in _tax}

# Validate that all framing categories are present in the loaded taxonomy.
invalid_cats = set(framing_map["framing_category"]) - valid_categories
if invalid_cats:
    raise ValueError(
        f"framing_map.csv contains categories not in framing_taxonomy.json: "
        f"{sorted(invalid_cats)}"
    )

# ── Build subject injection lookup (Pass A → __cat_* / __sub_*) ───────────────
inject_lookup = {}

for _, row in semantic_map.iterrows():
    if row.term not in vocab_df.index:
        continue
    if vocab_df.loc[row.term, "doc_freq"] > MAX_DOC_FREQ_FOR_INJECT:
        continue
    tokens = [f"__cat_{row.primary_category}__"]
    if pd.notna(row.get("subcategory")) and row.subcategory:
        tokens.append(f"__sub_{row.subcategory}__")
    inject_lookup.setdefault(row.term, []).extend(tokens)

# ── Build framing injection lookup (Pass B → __framing_*) ─────────────────────
for _, row in framing_map.iterrows():
    if row.term not in vocab_df.index:
        continue
    if vocab_df.loc[row.term, "doc_freq"] > MAX_DOC_FREQ_FOR_INJECT:
        # High-frequency terms only bypass the ceiling when they are a listed
        # load-bearing exception for their specific category.
        if ceiling_exceptions.get(row.term) != row.framing_category:
            continue
    inject_lookup.setdefault(row.term, []).append(f"__framing_{row.framing_category}__")

# ── Deduplicate tokens per term and apply min_inject_projects filter ──────────
# Filter is applied at the source-term level: a source term whose document
# frequency is below min_inject_projects is excluded from inject_lookup because
# it appears too rarely to contribute meaningful signal, even if it shares an
# injected token with higher-frequency terms.
# (See params.yaml → enrichment.min_inject_projects)
inject_lookup = {
    term: list(dict.fromkeys(tokens))
    for term, tokens in inject_lookup.items()
    if vocab_df.loc[term, "doc_freq"] >= MIN_INJECT_PROJECTS
}

unique_injected = {tok for toks in inject_lookup.values() for tok in toks}
print(f"Source terms with injections : {len(inject_lookup):,}")
print(f"Unique injected tokens        : {len(unique_injected):,}")

Ceiling exceptions : 31 terms from ceiling_exceptions.yaml
Source terms with injections : 1,564
Unique injected tokens        : 88


In [9]:
# ── Estimate injected token coverage before writing ───────────────────────────
# Counts how many projects will receive each injected token, and flags any
# that will fall below sql.min_doc_count after NB03's TF-IDF step.

min_doc_count = CFG["sql"]["min_doc_count"]

injected_token_counts: dict[str, int] = {}
for token_list in df["tokens"]:
    seen: set[str] = set()
    for t in token_list:
        if t in inject_lookup:
            seen.update(inject_lookup[t])
    for inj in seen:
        injected_token_counts[inj] = injected_token_counts.get(inj, 0) + 1

# Schema-stable even when inject_lookup is empty.
_inj_rows = [
    {"injected_token": tok, "project_count": cnt}
    for tok, cnt in injected_token_counts.items()
]
inj_freq_df = (
    pd.DataFrame(_inj_rows).sort_values("project_count", ascending=False)
    if _inj_rows
    else pd.DataFrame(columns=["injected_token", "project_count"])
)

will_survive = inj_freq_df[inj_freq_df["project_count"] >= min_doc_count]
will_drop    = inj_freq_df[inj_freq_df["project_count"] <  min_doc_count]

print(f"Injected tokens surviving min_doc_count={min_doc_count}: {len(will_survive)}")
if not will_drop.empty:
    print(f"WARNING: {len(will_drop)} injected tokens below threshold (will be filtered in NB03):")
    print(f"  {will_drop['injected_token'].tolist()}")
if not inj_freq_df.empty:
    print("\nTop injected tokens by project coverage:")
    print(will_survive.head(20).to_string(index=False))
else:
    print("No injections to preview (inject_lookup is empty).")

Injected tokens surviving min_doc_count=15: 88

Top injected tokens by project coverage:
                                 injected_token  project_count
                   __framing_chronic_scarcity__         507007
                    __framing_barrier_removal__         440101
              __framing_gratitude_orientation__         383011
             __framing_capacity_first_dignity__         372634
             __framing_joy_possibility_appeal__         365550
                __framing_inquiry_exploration__         315700
             __framing_durable_infrastructure__         303031
                 __framing_mandate_compliance__         220595
                        __framing_humble_plea__         220132
 __framing_self_regulation_executive_function__         217430
    __framing_mental_health_wellbeing_support__         206220
                 __framing_future_preparation__         205640
      __framing_collaborative_social_learning__         201687
          __framing_subgroup_

In [10]:
# ── Inject and write enriched parquet ────────────────────────────────────────
# inject_tokens() is defined in utils.py. Convert to plain Python lists first
# — parquet round-trips can produce numpy arrays that cause shape-mismatch
# errors inside inject_tokens.

df_enriched = df.copy()
df_enriched["tokens"] = df_enriched["tokens"].apply(
    lambda ts: inject_tokens(list(ts), inject_lookup)
)

orig_len          = df["tokens"].apply(len)
new_len           = df_enriched["tokens"].apply(len)
enriched_projects = (new_len > orig_len).sum()

print(f"Projects with injected tokens : {enriched_projects:,} ({enriched_projects/len(df):.1%})")
print(f"Mean tokens before injection  : {orig_len.mean():.1f}")
print(f"Mean tokens after injection   : {new_len.mean():.1f}")
print(f"Max injected per project      : {(new_len - orig_len).max()}")

out_path = OUT("prepared", "05_enriched.parquet")
df_enriched.to_parquet(out_path, index=False)
print(f"\nSaved → {out_path}")
print("\nTo use the enriched corpus in 03_insights_generation, load:")
print('  df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")')

# ── Stage manifest ────────────────────────────────────────────────────────────
stage_manifest_path = OUT("enrichment/metadata", "stage_manifest_02_semantic_enrichment.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status="success",
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/05_filtered.parquet",  "filtered_parquet"),
        artifact_meta(ROOT / "OUTPUTS/vocab/unigram_docfreq.csv",     "unigram_docfreq_csv"),
        artifact_meta(OUT("enrichment", "framing_taxonomy.json"),      "framing_taxonomy_json"),
    ],
    output_artifacts=[
        artifact_meta(OUT("enrichment", "semantic_map.csv"),           "semantic_map_csv"),
        artifact_meta(OUT("enrichment", "framing_map.csv"),            "framing_map_csv"),
        artifact_meta(OUT("enrichment", "framing_map_meta.json"),      "framing_map_meta_json"),
        artifact_meta(OUT("prepared",   "05_enriched.parquet"),        "enriched_parquet"),
    ],
    row_counts={
        "input_projects":        int(len(df)),
        "enriched_projects":     int(enriched_projects),
        "semantic_terms_mapped": int(len(semantic_map)),
        "framing_terms_mapped":  int(len(framing_map)),
        "inject_source_terms":   int(len(inject_lookup)),
    },
    key_params={
        "rare_doc_freq_ceiling":      CFG["enrichment"]["rare_doc_freq_ceiling"],
        "agg_distance_threshold":     CFG["enrichment"]["agg_distance_threshold"],
        "min_cluster_size":           CFG["enrichment"]["min_cluster_size"],
        "embedding_model":            CFG["enrichment"]["embedding_model"],
        "classify_batch_size":        CFG["enrichment"]["classify_batch_size"],
        "max_doc_freq_for_injection": CFG["enrichment"]["max_doc_freq_for_injection"],
        "min_inject_projects":        CFG["enrichment"]["min_inject_projects"],
    },
    warnings_path=WARNINGS_PATH,
)
print(f"Stage manifest → {stage_manifest_path}")

Projects with injected tokens : 2,164,870 (96.4%)
Mean tokens before injection  : 58.4
Mean tokens after injection   : 62.3
Max injected per project      : 18

Saved → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/prepared/05_enriched.parquet

To use the enriched corpus in 03_insights_generation, load:
  df = pd.read_parquet(ROOT / "OUTPUTS/prepared/05_enriched.parquet")
Stage manifest → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json
